In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("hw11.ipynb")

# Homework 11: Classification

**Helpful Resource:**

- [Python Reference](https://www.data8.org/datascience/reference-nb/datascience-reference.html): Cheat sheet of helpful array & table methods used in Data 111!

**Recommended Reading**: 

* [Classification](https://www.inferentialthinking.com/chapters/17/Classification.html)

Please complete this notebook by filling in the cells provided. Before you begin, execute the following cell to setup the notebook by importing some helpful libraries. Each time you start your server, you will need to execute this cell again.

For all problems that you must write explanations and sentences for, you **must** provide your answer in the designated space. **Moreover, throughout this homework and all future ones, please be sure to not re-assign variables throughout the notebook!** For example, if you use `max_temperature` in your answer to one question, do not reassign it later on. Otherwise, you will fail tests that you thought you were passing previously!


**Note: This homework has hidden tests on it. That means even though the tests may say 100% passed, it doesn't mean your final grade will be 100%. We will be running more tests for correctness once everyone turns in the homework.**


Directly sharing answers is not okay, but discussing problems with with other students is encouraged. 

You should start early so that you have time to get help if you're stuck.

In [ ]:
# Don't change this cell; just run it. 
import numpy as np
from datascience import *

# These lines do some fancy plotting magic
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import warnings
warnings.simplefilter('ignore')
from datetime import datetime

## 1. Classification: Condor or Turkey Vulture?

Welcome to Homework 11! This homework is about k-Nearest Neighbors classification (k-NN). Since this topic is covered in depth in Project 3, the purpose of this homework is to reinforce the basics of this method. You can and should reuse a lot of code that you wrote for Project 3 for this homework, or use code from this homework on Project 3!

### California Condor

<img src="Condor.jpg" width="300"/>

California Condor, or prey-go-neesh in Yurok, is a magnificent bird with the largest wingspan in North America. Their flight is a stable glide, at about 30 miles per hour. Reproduction is slow, with approximately one egg every two years. The young depend on their parents for over a year, and take 6-8 years to fully mature. These big birds nest in remote locations, but travel far in search of dead and decaying meat for food. They can live to be 50 years old or more!

California Condors reduce the potential for disease propagation by helping to remove the remains of large carcasses. They also have a specialized gastrointestinal microbiome that eliminates a variety of harmful bacteria and toxins, including anthrax, botulism, and cholera. Furthermore, their powerful bills can tear through tough hides, making the leftover carcasses available to smaller scavengers, such as turkey vultures, ravens, crows, raccoons, foxes, and skunks.  

Around 200 years ago condors were found throughout most of Western North America. By 1987, only a small community of 22 condors remained due to habitat loss, lead poisoning (by ingestion of meat killed with lead bullets), poaching, power-line accidents, and dichloro-diphenyl-trichloroethane (DDT) pesticides. In a conservation effort, the remaining birds were captured, bred in captivity, and eventually reintroduced to the wild. While the California Condor still faces many of the same threats, some conditions have improved. For example, the United States banned the use of DDT pesticides in 1972. Additionally, in 2019 California banned lead bullets for hunting.

### Yurok Condor Restoration Program

The Yurok Tribe is one of many indigenous cultures that considers condors sacred. Prey-go-neesh have been spiritually tied to the Yurok Hlkelonah — the cultural and ecological landscape — for thousands of years. Condors feature prominently in the Tribe’s origin narrative, and its feathers and songs are foundational components of Yurok World Renewal ceremonies. The Yurok Condor Restoration Program seeks to return the condor to the integral role it plays in healthy ecosystems and, in doing so, to renew and strengthen the spiritual lifeways of the many tribes who revere this majestic species. [Learn more about the Yurok connection to condors here](https://www.youtube.com/watch?v=01jINEEJZAc).

The Yurok Condor Restoration Program aims to restore the condor to Yurok Ancestral Territory and the Pacific Northwest. The Yurok Tribe Wildlife Program determined that there is strong potential for Prey-go-neesh to thrive in Northern California due to lower levels of lead and DDT and abundant diverse habitat. Thanks to the Yurok's Program, we now have Prey-go-neesh back in our community. Learn more about the [Yurok Condor Restoration Program here.](http://colab.research.google.com/drive/11a7BYorzBnV3XGNDT-c3KO-9IuzaWc44#scrollTo=2KwKULloSbLK&line=1&uniqifier=1)

### How to spot a Condor

Condors are easily confused with Turkey Vultures, but there are some ways to tell them apart:

<img src="condor_vs_turkey_vulture.jpg" width="1000"/>

### Using eBirds to check

If you think you might have spotted a Condor but you aren't sure, chances are that it was probably a Turkey Vulture. However, it might be helpful to check the eBird app! This is a community based science tool where anyone with the app can record their bird sightings. If other people in your area have recently seen a Condor, it is possible you saw one too. [You can learn more about the eBird app here](https://ebird.org/about/ebird-mobile/). (If you also want help identifying birds, you can download the companion app called [Merlin here](https://merlin.allaboutbirds.org/download/))

Data collected by eBird users is made publically available by the Cornell Lab of Ornithology. We retreived data for this assignment using the eBird API. You can learn more about APIs and how to work with them in DATA 271. For now, we will import the data as a csv as usual. 

Let's take a look at all the Condor and Turkey Vulture sitings in California that eBird users logged over the last month to test a k-NN algorithm to make species predictions. We want to classify birds as either Condor or Turkey Vulture using the latitude/longitude coordinates of the location they were observed.

In [ ]:
#Just run this cell! 
birds = Table.read_table('turvul_calcon_ca_11-2024.csv')
birds.show(5)

As usual, let's start by visualizing the data.

In [ ]:
# Just run this cell!
birds.scatter("lng", "lat", group="speciesCode")

The locations of the points on this scatter make a familiar shape - run the following cell to see them on a map of California.

In [ ]:
# Just run this cell!
def colors(species):
    if species == 'calcon':
        return 'blue'
    else:
        return 'red'
birds = birds.with_column('Color',birds.apply(colors,'speciesCode'))
t = Table().with_columns("lat", birds.column(7), 
                                      "lon", birds.column(8),
                                       "color", birds.column('Color'))
Circle.map_table(t, radius=5, fill_opacity=1)

**Question 1.1.** Let's begin implementing the k-Nearest Neighbors algorithm. Define the `distance` function, which takes in two arguments: an array of numerical features (`arr1`), and a different array of numerical features (`arr2`). The function should return the [Euclidean distance](https://en.wikipedia.org/wiki/Euclidean_distance) between the two arrays. Euclidean distance is often referred to as the straight-line distance formula that you may have learned previously. **(10 points)**



In [ ]:
def distance(arr1, arr2):
    ...

# Don't change/delete the code below in this cell
distance_example = distance(make_array(1, 2, 3), make_array(4, 5, 6))
distance_example

In [ ]:
grader.check("q1_1")

### Splitting the Dataset
We'll do two different kinds of things with the `birds` dataset:

1. We'll build a classifier using coordinates for which we know the associated label; this will teach it to recognize labels of similar coordinate values. This process is known as *training*.
2. We'll evaluate or *test* the accuracy of the classifier we build on data we haven't seen before.

As discussed in [Section 17.2](https://inferentialthinking.com/chapters/17/2/Training_and_Testing.html#training-and-testing), we want to use separate datasets for training and testing. As such, we split up our one dataset into two.

First let's calculate how many rows we want to use for each set, assuming we want to use approximately 75% of the data for training and 25% for testing. 

In [ ]:
# Just run this cell!
n = birds.num_rows
m = round(0.75*birds.num_rows)

**Question 1.2.** Next, let's split our dataset into a training set and a test set. 

Based on our previous calcations, let's create a training set with the first $m$ rows and a test set with the remaining rows. Remember that assignment to each group should be random, so we should shuffle the table first. **(10 points)**

*Hint:* As a first step we can shuffle all the rows, then use the `tbl.take` function to split up the rows for each table.

In [ ]:
shuffled_table = ...
train = ...
test = ...

print("Training set:\t",   train.num_rows, "examples")
print("Test set:\t",       test.num_rows, "examples")
train.show(5), test.show(5)

In [ ]:
grader.check("q1_2")

**Question 1.3.** Assign `features` to an array of column names (strings) of the features from the `birds` table. **(10 points)**

*Hint:* Which of the column names in the `birds` table are the features, and which of the column names correspond to the class we're trying to predict?

*Hint*: No need to modify any tables, just manually create an array of the feature names!

In [ ]:
features = ...
features

In [ ]:
grader.check("q1_3")

**Question 1.4.** Now define the `classify` function. This function should take in a `test_row` from a table like `test` and classify in using the k-Nearest Neighbors based on the correct `features` and the data in `train`. A refresher on k-Nearest Neighbors can be found [here](https://www.inferentialthinking.com/chapters/17/4/Implementing_the_Classifier.html). **(10 points)**


*Hint 1:* The `distance` function we defined earlier takes in arrays as input, so use the `row_to_array` function we defined for you to convert rows to arrays of features.

*Hint 2:* The skeleton code we provided iterates through each row in the training set.


In [ ]:
def row_to_array(row, features):
    """Converts a row to an array of its features."""
    arr = make_array()
    for feature in features:
        arr = np.append(arr, row.item(feature))
    return arr

def classify(test_row, k, train):
    test_row_features_array = row_to_array(test_row, features)
    distances = make_array()
    for train_row in train.rows:
        train_row_features_array = ...
        row_distance = ...
        distances = ...
    train_with_distances = ...
    nearest_neighbors = ...
    most_common_label = ...
    ...

# Don't modify/delete the code below
first_test = classify(test.row(0), 5, train)
first_test

In [ ]:
grader.check("q1_4")

**Question 1.5.0** Define the function `three_classify` that takes a `row` from `test` as an argument and classifies the row based on using 3-Nearest Neighbors. Use this function to find the `accuracy` of a 3-NN classifier on the `test` set. `accuracy` should be a proportion (not a percentage) of the birds that were correctly predicted. **(10 points)**


*Hint:* You should be using a function you just created!

*Note:* Usually before using a classifier on a test set, we'd classify first on a "validation" set, which we then can modify our training set again if need be, before actually testing on the test set. You don’t need to do that for this question, but you will learn about this more in Data 100.


In [ ]:
# This might take a few minutes to run! That is OK!
def three_classify(row):
    ...

test_with_prediction = ...
labels_correct = ...
accuracy = ...
accuracy


In [ ]:
grader.check("q1_5_0")

**Question 1.5.1** Now take an even closer look and calculate the Condor and Turkey Vulture accuracy separately. Use `calcon_accuracy` for the proportion of Condor observations in the test data that the predictor got correct. Use `turvul_accuracy` for the proportion of Turkey Vultures in the test data that the predictor got correct. **(5 points)**

In [ ]:
...
print(calcon_accuracy)
print(turvul_accuracy)

**Question 1.5.2** Most likely, your algorithm is much better at predicting Turkey Vultures than Condors, which is expected considering Condors are much more rare than Turkey Vultures. In addition, Turkey Vultures appear in many areas where Condors do not. Since this algorithm does not do very well at classifying Condors, we might wonder how it compares to a naive guess. 

What if instead of using k-NN, you simply guessed that *ALL* birds in the test data were Turkey Vultures. What would be the overall accuracy (`all_overall_accuracy`), Vulture accuracy (`all_turvul_accuracy`), and Condor accuracy (`all_calcon_accuracy`)? **(5 points)**

In [ ]:
...
print(all_overall_accuracy)
print(all_turvul_accuracy)
print(all_calcon_accuracy)

**Question 1.6.** If we used the entire `birds` table as the training set, what is the smallest value of k that would ensure that a k-Nearest Neighbor classifier would *always* predict `'turvul'` as the class? Assign the value to `k`. **(10 points)**


In [ ]:
num_condors = ...
k = ...
k

In [ ]:
grader.check("q1_6")

<!-- BEGIN QUESTION -->

**Question 1.7.** Why do we divide our data into a training and test set? What is the point of a test set, and why do we only want to use the test set once? Explain your answer in 3 sentences or less. **(10 points)**

*Hint:* Check out this [section](https://inferentialthinking.com/chapters/17/2/Training_and_Testing.html) in the textbook.


_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

**Question 1.8.** Why do we use an odd-numbered `k` in k-NN? Explain. **(10 points)**


_Type your answer here, replacing this text._

<!-- END QUESTION -->

**Question 1.9.0. Setup**

Thomas has devised a scheme for splitting up the test and training set. For each row from `birds`:
- Rows for Calfornia Condors have a 50% chance of being placed in the training set and 50% chance of being placed in the test set.  
- Rows for Turkey Vultures have a 80% chance of being placed in the training set and 20% chance of being placed in the test set.  


*Hint 1:* Calculate the number of Condors and Turkey Vultures in `birds`, and then calculate the proportion of each in the table.

*Hint 2:* Thomas' last name is Bayes. (So [18.1](https://inferentialthinking.com/chapters/18/1/More_Likely_than_Not_Binary_Classifier.html#bayes-rule) from the textbook may be helpful here!)

*Hint 3:* The following tree diagram may be helpful in Questions 1.9.1 and 1.9.2! Note that the proportion of condors and vultures in the diagram is not exact. You will want to calculate those exactly from your dataset. 

<img src="tree_diagram_condor_vulture.png" width="450">

**Question 1.9.1.** Given that a row is in the test set, what is the probability that it corresponds to a Condor? Assign that probability to `prob_con`. **(5 points)**


In [ ]:
condor_proportion = ...
vulture_proportion = ...
prob_con = ...
prob_con

In [ ]:
grader.check("q1_9_1")

**Question 1.9.2.** Given that a row is a Condor, what is the probability that the Condor is in the test set? Assign that probability to `prob_test`. **(5 points)**


In [ ]:
prob_test = ...
prob_test

In [ ]:
grader.check("q1_9_2")

## (OPTIONAL): k-NN for Non-Binary Classification

**THIS IS NOT IN SCOPE**. There are no autograder tests for this or code for you to write. It just relies on the function `classify` in Question 1.4. Go ahead and read through this section and run the following cells!

In this class, we have taught you how to use the k-NN algorithm to classify data as one of two classes. However, much of the data you will encounter in the real world will not fall nicely into one of two categories. 

**How can we classify data with non-binary classes?** It turns out we can still use k-NN! That is, we find the distance between a point and all its neighbors, find the nearest neighbors, and take a majority vote among the neighbors to determine this point's class. 

The only difference is that now the neighboring points have more than two possible classes. This does introduce difficulty because now we have no way of guaranteeing that we will not encounter ties between classes. In the case that we do encounter a tie, we can just arbitrarily choose one of the classes.

In fact, you don't even have to modify the code you wrote before at all to enable multi-class classification!

Let's add some more data to our train table, this time for another type of bird, the Golden Eagle.

In [ ]:
#This is how I downloaded the data from the eBird API in case you ever want to know how to do it yourself
# goleag = get_species_observations(token="add your token here",species='goleag',area='US-CA',back=30)
# df = pd.DataFrame(goleag)
# df.to_csv('goleag_ca_20231212_20240111.csv')
goleag = Table.read_table('goleag_ca.csv')
goleag = goleag.with_column('Color','yellow').drop(0)
birds_multi = birds.with_rows(goleag.rows)

colors = {"calcon":"blue", "turvul":"red", "goleag":"yellow"}
t = Table().with_columns("lat", birds_multi.column(7), 
                                      "lon", birds_multi.column(8), 
                                      "color", birds_multi.column('Color')
                        )
Circle.map_table(t, radius=5, fill_opacity=1)

In [ ]:
# find condor and test classifier
condors = birds_multi.where("speciesCode",are.equal_to("calcon"))
classify(birds_multi.row(4122), 5, birds_multi)

In [ ]:
# find vulture and test classifier
vulture = birds_multi.where("speciesCode",are.equal_to("turvul"))
classify(birds_multi.row(0), 5, birds_multi)

In [ ]:
# find golden eagle and test classifier
geagle = birds_multi.where("speciesCode",are.equal_to("goleag"))
classify(birds_multi.row(4173), 5, birds_multi)

Our classifier can classify rows as belonging to one of three classes! Do you think this method will help with differentiating Golden Eagles and Turkey Vultures? Why or why not?

Classification is one of the most important fields in statistics, data science, and machine learning. There are thousands of different classification algorithms and modifications of algorithms! There are many that you'll learn if you continue down the path of becoming a data scientist!

You're all done with Homework 11! :,)

**Important submission steps:** 
1. Run the tests and verify that they all pass.
2. Choose **Save Notebook** from the **File** menu, then **run the final cell**. 
3. Click the link to download the zip file.
4. Then submit the zip file to the corresponding assignment to Canvas. 

**It is your responsibility to make sure your work is saved before running the last cell.**

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)